In [2]:
import os, sys

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PYSPARK_PYTHON'] = sys.executable

print('Python:', sys.version)
print('JAVA_HOME:', os.environ.get('JAVA_HOME'))

Python: 3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DisasterStreamToMongo") \
    .config("spark.mongodb.write.connection.uri",
            "mongodb://localhost:27017/ccfraud_db.creditCard") \
    .config("spark.jars.packages",
            "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()

spark.version

:: loading settings :: url = jar:file:/home/vboxuser/spark-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vboxuser/.ivy2/cache
The jars for the packages stored in: /home/vboxuser/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3f559e03-4624-4b63-a583-af23e4aec13a;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.12;10.3.0 in central
	found org.mongodb#mongodb-driver-sync;4.8.2 in central
	[4.8.2] org.mongodb#mongodb-driver-sync;[4.8.1,4.8.99)
	found org.mongodb#bson;4.8.2 in central
	found org.mongodb#mongodb-driver-core;4.8.2 in central
	found org.mongodb#bson-record-codec;4.8.2 in central
:: resolution report :: resolve 2394ms :: artifacts dl 12ms
	:: modules in use:
	org.mongodb#bson;4.8.2 from central in [default]
	org.mongodb#bson-record-codec;4.8.2 from central in [default]
	org.mongodb#mongodb-driver-core;4.8.2 from central in [default]
	org.mongodb#mongodb-driver-sync;4.8.2 from central in [default]
	org.mongodb.spark#mongo-spark-conne

'3.5.0'

In [5]:
#Read csv file
df = spark.read.csv("/home/vboxuser/Big Data Systems Design/Final Project/Fraud2/creditcard.csv",
                    header = True,
                    inferSchema = True)
df.show(8)


+----+------------------+-------------------+------------------+------------------+-------------------+-------------------+--------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------------+-------------------+-------------------+------------------+------------------+-------------------+--------------------+-------------------+------+-----+
|Time|                V1|                 V2|                V3|                V4|                 V5|                 V6|                  V7|                V8|                V9|                V10|               V11|               V12|               V13|               V14|               V15|                V16|                V17|                V18|                V19|                V20|                 V21|           

In [6]:
df.printSchema()

root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = true)
 |-- V28: double (nulla

In [7]:
# Show counts for: rows and columns
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 284807
Columns: 31


In [6]:
print(df.columns)

['date', 'country', 'disaster_type', 'severity_index', 'casualties', 'economic_loss_usd', 'response_time_hours', 'aid_amount_usd', 'response_efficiency_score', 'recovery_days', 'latitude', 'longitude']


In [8]:
# import all functions
from pyspark.sql.functions import *

#Identify Columns of Interest
df_basic = df.select(
    "Time",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
    "V7",
    "V8",
    "V9",
    "V10",
    "V11",
    "V12",
    "V13",
    "V14",
    "V15",
    "V16",
    "V17",
    "V18",
    "V19",
    "V20",
    "V21",
    "V22",
    "V23",
    "V24",
    "V25",
    "V26",
    "V27",
    "V28",
    "Amount",
    "Class"
)

df_basic.show(5)

+----+------------------+-------------------+----------------+------------------+-------------------+-------------------+-------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+--------------------+-------------------+------------------+------------------+------------------+------------------+--------------------+-------------------+------+-----+
|Time|                V1|                 V2|              V3|                V4|                 V5|                 V6|                 V7|                V8|                V9|                V10|               V11|               V12|               V13|               V14|               V15|               V16|               V17|                V18|               V19|                V20|                 V21|                V22|     

In [9]:
df_clean = df_basic

In [10]:
df_clean.printSchema()

root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = true)
 |-- V28: double (nulla

In [12]:
# Create a single row data frame showing the number of null values in each column of interest
null_counts = df_clean.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_basic.columns])

#print full results with column names
null_counts.show(truncate=False)

[Stage 13:===========================================>              (3 + 1) / 4]

+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|Time|V1 |V2 |V3 |V4 |V5 |V6 |V7 |V8 |V9 |V10|V11|V12|V13|V14|V15|V16|V17|V18|V19|V20|V21|V22|V23|V24|V25|V26|V27|V28|Amount|Class|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|0   |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0     |0    |
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+



In [13]:
# Spark ML requires all features be in one column called features 
from pyspark.ml.feature import VectorAssembler

featureCols = [
    "Time","V1","V2","V3","V4","V5","V6","V7","V8","V9","V10",
    "V11","V12","V13","V14","V15","V16","V17","V18","V19",
    "V20","V21","V22","V23","V24","V25","V26","V27","V28","Amount"
]

assembler = VectorAssembler(
    inputCols = featureCols,
    outputCol = "features"
)

dfFeatures = assembler.transform(df_clean)

dfModel = dfFeatures.select("features","Class")
dfModel.show(5)

+--------------------+-----+
|            features|Class|
+--------------------+-----+
|[0.0,-1.359807133...|    0|
|[0.0,1.1918571113...|    0|
|[1.0,-1.358354061...|    0|
|[1.0,-0.966271711...|    0|
|[2.0,-1.158233093...|    0|
+--------------------+-----+
only showing top 5 rows



In [14]:
#Split data set
trainDf, testDf = dfModel.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", trainDf.count())
print("Testing rows:", testDf.count())

Training rows: 227792


[Stage 20:=============================>                            (2 + 2) / 4]

Testing rows: 57015


In [15]:
#Logistic Regression Model
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="Class"
)

lrModel = lr.fit(trainDf)

lrPredictions = lrModel.transform(testDf)

lrPredictions.select("Class","prediction","probability").show(5)

[Stage 87:>                                                         (0 + 1) / 1]

+-----+----------+--------------------+
|Class|prediction|         probability|
+-----+----------+--------------------+
|    0|       0.0|[0.99968071195413...|
|    0|       0.0|[0.99954452099332...|
|    0|       0.0|[0.99906059772620...|
|    0|       0.0|[0.99861566320023...|
|    0|       0.0|[0.99995755398060...|
+-----+----------+--------------------+
only showing top 5 rows



In [16]:
# Random Forest Model
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="Class",
    numTrees=100
)

rfModel = rf.fit(trainDf)

rfPredictions = rfModel.transform(testDf)

rfPredictions.select("Class","prediction","probability").show(5)

[Stage 105:>                                                        (0 + 1) / 1]

+-----+----------+--------------------+
|Class|prediction|         probability|
+-----+----------+--------------------+
|    0|       0.0|[0.99973668923964...|
|    0|       0.0|[0.99977466495691...|
|    0|       0.0|[0.99926930505317...|
|    0|       0.0|[0.99969494707397...|
|    0|       0.0|[0.99977466495691...|
+-----+----------+--------------------+
only showing top 5 rows



In [17]:
#SVM Model
from pyspark.ml.classification import LinearSVC

svm = LinearSVC(
    featuresCol="features",
    labelCol="Class",
    maxIter=10
)

svmModel = svm.fit(trainDf)

svmPredictions = svmModel.transform(testDf)

svmPredictions.select("Class","prediction").show(5)

[Stage 135:>                                                        (0 + 1) / 1]

+-----+----------+
|Class|prediction|
+-----+----------+
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
+-----+----------+
only showing top 5 rows



In [18]:
#Evaluate Models
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="Class",
    metricName="areaUnderROC"
)

lrAuc = evaluator.evaluate(lrPredictions)
rfAuc = evaluator.evaluate(rfPredictions)
svmAuc = evaluator.evaluate(svmPredictions)

print("Logistic Regression AUC:", lrAuc)
print("Random Forest AUC:", rfAuc)
print("SVM AUC:", svmAuc)

Logistic Regression AUC: 0.9559664190439364
Random Forest AUC: 0.9665580649777441
SVM AUC: 0.9600646369385484


In [11]:
#distinct values in disaster_type
df_clean.select("disaster_type").distinct().orderBy("disaster_type").show()

+-----------------+
|    disaster_type|
+-----------------+
|          Drought|
|       Earthquake|
|     Extreme Heat|
|            Flood|
|        Hurricane|
|        Landslide|
|      Storm Surge|
|          Tornado|
|Volcanic Eruption|
|         Wildfire|
+-----------------+



In [12]:
# Create a year/month column from date for yearly trend analysis

df_clean= df.withColumn("DisasterYear",year(col("date")))\
.withColumn("DisasterMonth",month(col("date")))

df_clean.columns

['date',
 'country',
 'disaster_type',
 'severity_index',
 'casualties',
 'economic_loss_usd',
 'response_time_hours',
 'aid_amount_usd',
 'response_efficiency_score',
 'recovery_days',
 'latitude',
 'longitude',
 'DisasterYear',
 'DisasterMonth']

## Analysis Using Spark SQL

In [13]:
#Using Spark SQL for basic analysis

# Create or Replace view
df_clean.createOrReplaceTempView("disasterVW")

#Yearly Disaster Count Trends
spark.sql("""
select 
    year(date),
    count(*) as TotalDisasters
from disasterVW
group by year(date)
order by TotalDisasters desc""").show()


+----------+--------------+
|year(date)|TotalDisasters|
+----------+--------------+
|      2021|          7271|
|      2020|          7247|
|      2023|          7139|
|      2022|          7130|
|      2019|          7113|
|      2024|          7086|
|      2018|          7014|
+----------+--------------+



In [14]:
#Show Average Severity by Disaster Type

spark.sql("""
select 
    disaster_type,
    avg(severity_index) as AvgSev
from disasterVW
group by disaster_type
order by AvgSev desc""").show()

+-----------------+------------------+
|    disaster_type|            AvgSev|
+-----------------+------------------+
|          Tornado| 5.081945738003639|
|Volcanic Eruption| 5.058508930363244|
|         Wildfire|5.0466027452563536|
|        Landslide| 5.028793372319693|
|      Storm Surge|5.0238712910986205|
|        Hurricane|5.0160735705717645|
|     Extreme Heat| 5.007490501899616|
|            Flood|  4.97089104981148|
|       Earthquake| 4.970311760063148|
|          Drought| 4.953794934640533|
+-----------------+------------------+



In [15]:
# Show Economic Loss By Country

spark.sql("""
select 
    country,
    sum(economic_loss_usd) as CountyEconomicLoss
from disasterVW
group by country
order by CountyEconomicLoss desc""").show()


+-------------+--------------------+
|      country|  CountyEconomicLoss|
+-------------+--------------------+
|       Brazil|1.320586680846997...|
|   Bangladesh|1.302648932460999...|
| South Africa|1.295759070633001E10|
|        Italy|1.290835925490997...|
|        China|1.285749705428999...|
|       Greece|1.282855191831997...|
|      Germany|1.273711626128000...|
|    Australia|   1.269431912943E10|
|      Nigeria|1.268575095718000...|
|       Turkey|1.268118956244000...|
|        India|1.265547203346000...|
|  Philippines|1.261609141128002...|
|    Indonesia|1.261571084954998...|
|        Chile|1.256236155257000...|
|        Japan|1.252109347134999...|
|        Spain|   1.251307350536E10|
|United States|1.235948222631003E10|
|       Canada|1.235621431991000...|
|       Mexico|1.233096175275999...|
|       France|   1.231648031345E10|
+-------------+--------------------+



## Analysis using Dataframe

In [16]:
#Yearly Disaster Count Trends

disasterCounts = (df_clean
.groupBy(col("DisasterYear").alias("DisasterYear"))
.count().alias("CountYearlyDisaster")
.orderBy("count",ascending=False)
)

disasterCounts.show(10, truncate=False)

+------------+-----+
|DisasterYear|count|
+------------+-----+
|2021        |7271 |
|2020        |7247 |
|2023        |7139 |
|2022        |7130 |
|2019        |7113 |
|2024        |7086 |
|2018        |7014 |
+------------+-----+



In [17]:
#Show Average Severity by Disaster Type

AvgSevByDisType = (df_clean
.groupBy(col("disaster_type"))
.agg(round(avg("severity_index"),2).alias("AvgSevIndex"))
.orderBy("AvgSevIndex",ascending=False)
)

AvgSevByDisType.show(10, truncate=False)

+-----------------+-----------+
|disaster_type    |AvgSevIndex|
+-----------------+-----------+
|Tornado          |5.08       |
|Volcanic Eruption|5.06       |
|Wildfire         |5.05       |
|Landslide        |5.03       |
|Hurricane        |5.02       |
|Storm Surge      |5.02       |
|Extreme Heat     |5.01       |
|Earthquake       |4.97       |
|Flood            |4.97       |
|Drought          |4.95       |
+-----------------+-----------+



In [18]:
# Rank disaster_type within each country by severity_index

from pyspark.sql.window import Window #allows calculation across groups
from pyspark.sql.functions import row_number #pseudo row number

PartitionOverCountry = Window.partitionBy("country").orderBy(col("severity_index")\
    .desc())

df_rank = df_clean.withColumn(
    "Severity_Rank_By_Country",
    row_number().over(PartitionOverCountry)
)

df_rank.show(5)


+----------+-------+-----------------+--------------+----------+-----------------+-------------------+--------------+-------------------------+-------------+--------+---------+------------+-------------+------------------------+
|      date|country|    disaster_type|severity_index|casualties|economic_loss_usd|response_time_hours|aid_amount_usd|response_efficiency_score|recovery_days|latitude|longitude|DisasterYear|DisasterMonth|Severity_Rank_By_Country|
+----------+-------+-----------------+--------------+----------+-----------------+-------------------+--------------+-------------------------+-------------+--------+---------+------------+-------------+------------------------+
|2020-11-28|  China|       Earthquake|          10.0|       196|       9191143.08|               1.23|     548934.07|                    99.76|           99| -48.591|  146.502|        2020|           11|                       1|
|2018-02-19|  China|     Extreme Heat|          10.0|       170|    1.165523255E7|  

In [19]:
# Show the severity of disaster over the years

AvgSevByYear= (df_clean
.groupBy(col("DisasterYear"))
.agg(round(avg("severity_index"),2).alias("AvgSevIndex"))
.orderBy("DisasterYear",ascending=True)
)

AvgSevByYear.show(10, truncate=False)

+------------+-----------+
|DisasterYear|AvgSevIndex|
+------------+-----------+
|2018        |5.03       |
|2019        |5.01       |
|2020        |5.05       |
|2021        |4.99       |
|2022        |4.99       |
|2023        |5.01       |
|2024        |5.03       |
+------------+-----------+



In [20]:
# Show if response_efficiency_score is getting better over the years
AvgResEffYear= (df_clean
.groupBy(col("DisasterYear"))
.agg(round(avg("response_efficiency_score"),2).alias("AvgResEff"))
.orderBy("DisasterYear",ascending=True)
)

AvgResEffYear.show(10, truncate=False)


+------------+---------+
|DisasterYear|AvgResEff|
+------------+---------+
|2018        |87.58    |
|2019        |87.51    |
|2020        |87.63    |
|2021        |87.48    |
|2022        |87.54    |
|2023        |87.68    |
|2024        |87.59    |
+------------+---------+



In [21]:
# Show Disaster Count By Country

from pyspark.sql.functions import count

DisasterCntByCountry= (df_clean
.groupBy(col("country"))
.agg(count("*").alias("CountDisasterByCountry"))
.orderBy("CountDisasterByCountry",ascending=False)
)

DisasterCntByCountry.show(10, truncate=False)

+----------+----------------------+
|country   |CountDisasterByCountry|
+----------+----------------------+
|Brazil    |2591                  |
|Australia |2563                  |
|Turkey    |2554                  |
|Bangladesh|2553                  |
|Spain     |2543                  |
|China     |2539                  |
|Chile     |2529                  |
|Nigeria   |2528                  |
|Germany   |2526                  |
|India     |2509                  |
+----------+----------------------+
only showing top 10 rows



In [22]:
# Show Economic Loss By Country
DisasterCntByCountry= (df_clean
.groupBy(col("country"))
.agg(sum(col("economic_loss_usd")).alias("EconLossByCountry"))
.orderBy("EconLossByCountry",ascending=False)
)

DisasterCntByCountry.show(10, truncate=False)

+------------+---------------------+
|country     |EconLossByCountry    |
+------------+---------------------+
|Brazil      |1.3205866808469978E10|
|Bangladesh  |1.3026489324609993E10|
|South Africa|1.295759070633001E10 |
|Italy       |1.2908359254909971E10|
|China       |1.2857497054289993E10|
|Greece      |1.2828551918319979E10|
|Germany     |1.2737116261280005E10|
|Australia   |1.269431912943E10    |
|Nigeria     |1.2685750957180004E10|
|Turkey      |1.2681189562440006E10|
+------------+---------------------+
only showing top 10 rows



In [23]:
# for each row in df_clean add a Country Loss column

countryLoss = (df_clean
.groupBy(col("country"))
#.sum("economic_loss_usd")
.agg(sum(col("economic_loss_usd")).alias("EconLossByCountry"))
#.withColumnRenamed("sum(economic_loss_usd)","total_loss")
)

df_join = df_clean.join(countryLoss, on="country", how="left") #left join return all column from df_clean

In [24]:
df_join.show(5)

+-------------+----------+-------------+--------------+----------+-----------------+-------------------+--------------+-------------------------+-------------+--------+---------+------------+-------------+--------------------+
|      country|      date|disaster_type|severity_index|casualties|economic_loss_usd|response_time_hours|aid_amount_usd|response_efficiency_score|recovery_days|latitude|longitude|DisasterYear|DisasterMonth|   EconLossByCountry|
+-------------+----------+-------------+--------------+----------+-----------------+-------------------+--------------+-------------------------+-------------+--------+---------+------------+-------------+--------------------+
|       Brazil|2021-01-31|   Earthquake|          5.99|       111|       7934365.71|              15.62|     271603.79|                    83.21|           67| -30.613| -122.557|        2021|            1|1.320586680846997...|
|       Brazil|2018-12-23| Extreme Heat|          6.53|       100|       8307648.99|        

## SPARK STRUCTURED STREAMING

In [25]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# C1- Create a streaming source to simulate disaster data

df_streamrate = (
    spark.readStream
    .format("rate")
    .option("rowspersecond",5)
    .load()
)

In [26]:
# Pre-defined values
disaster_types = array(
    lit("Drought"),
    lit("Earthquake"),
    lit("Extreme Heat"),
    lit("Flood"),
    lit("Hurricane"),
    lit("Landslide"),
    lit("Storm Surge"),
    lit("Tornado"),
    lit("Volcanic Eruption"),
    lit("Wildfire")
)

countries = array(
    lit("USA"),
    lit("Canada"),
    lit("India"),
    lit("Brazil"),
    lit("Japan"),
    lit("Indonesia"),
    lit("Mexico"),
    lit("Philippines")
)

In [27]:
DisasterStream = df_streamrate.select(
    
    # Use generated timestamp as date
    col("timestamp").alias("date"),
    
    # Random country
    element_at(countries, (rand()*8 + 1).cast("int")).alias("country"),
    
    # Random disaster type
    element_at(disaster_types, (rand()*10 + 1).cast("int")).alias("disaster_type"),

    # Severity 1–10
    (rand()*9 + 1).cast("int").alias("severity_index"),
    
    # Casualties 1–1000
    (rand()*999 + 1).cast("int").alias("casualties"),
    
    # Economic loss random up to 10M
    (rand()*10000000).cast("double").alias("economic_loss_usd"),
    
    # Response time < 63 hours
    (rand()*62 + 1).cast("int").alias("response_time_hours"),

    # Aid amount random up to 5M
    (rand()*5000000).cast("double").alias("aid_amount_usd"),
    
    # Response efficiency 1–100 score
    (rand()*99 + 1).cast("double").alias("response_efficiency_score")
)

In [28]:
# Add derived columns
DisasterStream = DisasterStream \
    .withColumn("DisasterYear", year(col("date"))) \
    .withColumn("DisasterMonth", month(col("date")))

In [29]:
def write_to_mongo(batch_df, batch_id):
    batch_df.write \
        .format("mongodb") \
        .mode("append") \
        .save()

mongo_query = DisasterStream.writeStream \
    .foreachBatch(write_to_mongo) \
    .option("checkpointLocation", "/tmp/mongo_checkpoint") \
    .outputMode("append") \
    .start()

26/03/04 18:34:53 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [30]:
spark.streams.active

In [31]:
mongo_query.status

{'message': 'Processing new data',
 'isDataAvailable': True,
 'isTriggerActive': True}

In [32]:
# Using local filesystem to simulate distributed storage
hdfs_query = DisasterStream.writeStream \
    .format("parquet") \
    .option("path", "/home/vboxuser/Big Data Systems Design/Individual Project/disaster_archive") \
    .option("checkpointLocation", "/tmp/hdfs_checkpoint") \
    .outputMode("append") \
    .start()

26/03/04 18:35:04 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/03/04 18:35:08 ERROR FileFormatWriter: Aborting job 60c764a1-2fca-4086-ac27-b073afb46892.
org.apache.spark.SparkFileNotFoundException: [BATCH_METADATA_NOT_FOUND] Unable to find batch /home/vboxuser/Big Data Systems Design/Individual Project/disaster_archive/_spark_metadata/79.compact.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.batchMetadataFileNotFoundError(QueryExecutionErrors.scala:1848)
	at org.apache.spark.sql.execution.streaming.HDFSMetadataLog.applyFnToBatchByStream(HDFSMetadataLog.scala:194)
	at org.apache.spark.sql.execution.streaming.CompactibleFileStreamLog.applyFnInBatch(CompactibleFileStreamLog.scala:207)
	at org.apache.spark.sql.execution.streaming.CompactibleFileStreamLog.foreachInBatch(CompactibleFileStreamLog.scala:193)
	at org.apache.spark.sql.execution.streaming.CompactibleFileStreamLog.$anonfun$compact$3(Compact

In [33]:
#stop active stream
for q in spark.streams.active:
    q.stop()

In [34]:
#spark.version
mongo_query.stop()


In [35]:
hdfs_query.stop()